# KinoCompass – итоговый тест: EASE

**Тест – один раз.** EASE с λ = 5000 (выбран на валидации) переобучается на всём до 14.10.2022 и проверяется на последнем годе. Модель не меняется.

На тесте 22 309 кандидатов: матрица «фильм × фильм» в float64 – 4 ГБ. Обращение – на GPU через разложение Холецкого полосами по 2048 столбцов (в памяти: разложение 4 ГБ + итоговая матрица 2 ГБ + полоса), а **до расчёта** программа проверяет свободную видеопамять и сразу останавливается с понятным сообщением, если её не хватает.

Состав (`python evaluate.py final_ease`): популярные за последний год (для парных сравнений) и EASE λ = 5000.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `final_ease_test.zip` – его нужно положить в `model/reports/`.

Время на T4 (оценка): загрузка набора и сценарии – около 3 минут, XᵀX и обращение – 2–5 минут, проверка – около 2 минут; всего около 10 минут.

In [1]:
# итоговый тест – один раз (модели выбраны на валидации, шаг 6в)
PERIOD = "test"

In [2]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

Tesla T4 14.6 ГБ


In [3]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

/content
07beafc (grafted, HEAD -> main, origin/main, origin/HEAD) Разделил итоговый тест в Colab: Mult-VAE отдельно, EASE полосами на GPU с проверкой памяти
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [4]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py final_ease

/content/KinoCompass/model
скачиваю https://files.grouplens.org/datasets/movielens/ml-32m.zip в память
  0 % (1 из 227 МБ)
  25 % (57 из 227 МБ)
  50 % (114 из 227 МБ)
  75 % (171 из 227 МБ)
  100 % (227 из 227 МБ)
  ratings: 32 000 204 строк
  tags: 2 000 072 строк
  movies: 87 585 строк
  links: 87 585 строк
кандидаты: candidates, 22309 фильмов
сценарии: {'cold5': {'people': 4258, 'median_inputs': 5, 'median_targets': 44}, 'cold10': {'people': 4252, 'median_inputs': 10, 'median_targets': 41}, 'warm': {'people': 5413, 'median_inputs': 320, 'median_targets': 8}}
cold5 popular_year ndcg@10 0.28512
cold10 popular_year ndcg@10 0.24941
warm popular_year ndcg@10 0.09724
EASE: на GPU свободно 14.4 ГиБ, нужно 7.9 – считаю полосами
cold5 ease_l5000 ndcg@10 0.34266
cold10 ease_l5000 ndcg@10 0.32247
warm ease_l5000 ndcg@10 0.12477
этапы, с: {'load': 20.5, 'scenarios': 116.9, 'fit': 159.1, 'evaluate': 65.1}
сохранено reports/final_ease/test.json за 362.0 с


In [5]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/final_ease_{PERIOD}.zip reports/final_ease && ls -la reports/final_ease
files.download(f"/content/final_ease_{PERIOD}.zip")

total 108
drwxr-xr-x  2 root root  4096 Nov 11 11:28 .
drwxr-xr-x 13 root root  4096 Nov 11 11:28 ..
-rw-r--r--  1 root root  4736 Nov 11 11:28 test.json
-rw-r--r--  1 root root 94094 Nov 11 11:28 test_users.csv.gz
